# Round trip: every assembled data source for one tile / year

Reconstructs a raster for **every column** of the assembled 1 km panel for a
single `(ix, iy)` tile and a single `year`, straight from `pixel_id`, and lays
them all out in one figure. This is the round-trip check that the flat panel can
be folded back to the pixel grid it came from.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from odc.geo.xr import ODCExtensionDa  # noqa: F401  (registers the .odc accessor)
from odc.geo import GeoboxTiles

In [ ]:
# --- parameters --------------------------------------------------------------
PROJECT = "/scicore/home/meiera/schulz0022/projects/growth-and-temperature"
DATA = f"{PROJECT}/data_nobackup"

ix, iy = 0, 0          # tile index into the 2048 x 2048 GeoboxTiles grid
year = 2013            # panel year to plot (ignored for time-invariant columns)
grid_label = "1km"
shake_label = "base"

assembled_tile = (
    f"{DATA}/assembled/grid={grid_label}/shake={shake_label}"
    f"/ix={ix}/iy={iy}/data.parquet"
)
land_mask_zarr = f"{DATA}/misc/processed/stage_2/osm/land_mask.zarr"

In [ ]:
# --- load the panel tile + the geobox it was cut from ----------------------
parquet_tile = pd.read_parquet(assembled_tile)
land_mask = xr.open_zarr(land_mask_zarr, consolidated=False)

tile_size = (2048, 2048)
tile = GeoboxTiles(land_mask.odc.geobox, tile_size)[ix, iy]

print("panel rows :", len(parquet_tile))
print("columns    :", list(parquet_tile.columns))
if "year" in parquet_tile.columns:
    yrs = np.sort(parquet_tile["year"].dropna().unique())
    print("years      :", yrs.min(), "..", yrs.max(), f"({len(yrs)} unique)")

In [ ]:
# --- pixel_id -> (longitude, latitude) for this tile ----------------------
h, w = tile.shape
local_pixel_ids = np.arange(h * w, dtype="uint32").reshape((h, w))
pixel_id_matrix = (
    (np.uint64(ix) << 48) | (np.uint64(iy) << 32) | local_pixel_ids.astype(np.uint64)
)

lon, lat = np.meshgrid(
    tile.coords["longitude"].values, tile.coords["latitude"].values
)
conversion_df = pd.DataFrame(
    {
        "pixel_id": pixel_id_matrix.reshape(-1),
        "longitude": lon.reshape(-1),
        "latitude": lat.reshape(-1),
    }
)

In [ ]:
# --- pick the columns to plot -------------------------------------------------
id_cols = {"pixel_id", "year", "ix", "iy", "longitude", "latitude"}
value_cols = [c for c in parquet_tile.columns if c not in id_cols]

panel = parquet_tile
if "year" in panel.columns:
    panel = panel[panel["year"] == year]
    if panel.empty:
        raise ValueError(f"no panel rows for year={year}")

# drop columns that are entirely missing for this tile/year
value_cols = [c for c in value_cols if panel[c].notna().any()]

# object / category columns -> integer codes so they still render
encoded = panel[["pixel_id"]].copy()
cat_cols = []
for c in value_cols:
    s = panel[c]
    if s.dtype == object or str(s.dtype).startswith("category"):
        encoded[c] = pd.factorize(s)[0].astype("float32")
        encoded.loc[s.isna().values, c] = np.nan
        cat_cols.append(c)
    else:
        encoded[c] = s.astype("float32")

print(f"{len(value_cols)} columns to plot; categorical (code-encoded): {cat_cols}")

In [ ]:
# --- fold every column back onto the tile grid ------------------------------
grid = (
    conversion_df.merge(encoded, on="pixel_id", how="left")
    .set_index(["latitude", "longitude"])[value_cols]
    .to_xarray()
)
grid = grid.odc.assign_crs(4326)
grid

In [ ]:
# --- one imshow per source -------------------------------------------------
ncols = 3
nrows = int(np.ceil(len(value_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.4 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, col in zip(axes, value_cols):
    da = grid[col]
    cmap = "tab20" if col in cat_cols else "viridis"
    da.plot.imshow(ax=ax, robust=True, cmap=cmap, add_labels=False)
    ax.set_title(col + (" (codes)" if col in cat_cols else ""), fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

for ax in axes[len(value_cols):]:
    ax.set_visible(False)

fig.suptitle(f"assembled 1km panel  -  tile ix={ix} iy={iy}  -  year {year}", y=1.0)
fig.tight_layout()